In [ ]:
print("RAG IMPLEMENTATION")

In [ ]:
%pip install python-dotenv pydantic chromadb tqdm numpy scikit-learn plotly sentence-transformers

In [ ]:
# %pip install chromadb

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os
import importlib.util

# if importlib.util.find_spec("sentence_transformers") is None:
#     %pip install sentence-transformers

from sentence_transformers import SentenceTransformer


In [ ]:
load_dotenv(override=True)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "gpt-oss:120b-cloud" 
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
KNOWLEDGE_BASE_PATH = Path("knowledge_base")
AVERAGE_CHUNK_SIZE = 500
openai = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)

In [ ]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")
    
    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)
    
class Chunks(BaseModel):
    chunks: list[Chunk]


In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [ ]:
documents = fetch_documents()

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of the company described in the Knowledge Base.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company described in the Knowledge Base.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""


In [ ]:
print(make_prompt(documents[0]))

In [ ]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [ ]:
make_messages(documents[0])

In [ ]:
def process_document(document):
    messages = make_messages(document)
    messages[0]["content"] += "\n\nRespond ONLY with a valid JSON object matching this schema, no markdown or backticks:\n" + Chunks.model_json_schema().__str__()
    
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        response_format={"type": "json"}
    )
    
    raw = response.choices[0].message.content.strip()
    doc_as_chunks = Chunks.model_validate_json(raw)
    return [chunk.as_result(document) for chunk in doc_as_chunks.chunks]

In [ ]:
process_document(documents[1])

In [ ]:
def create_chunks(document):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [ ]:
chunks = create_chunks(documents)

In [ ]:
print(chunks)
print(len(chunks))

In [ ]:
print(chunks[7])

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    vectors = embedding_model.encode(texts).tolist()

    collection = chroma.get_or_create_collection(collection_name)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
create_embeddings(chunks)

In [ ]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)

print(collection.count())

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text retrieved from the company Knowledge Base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = openai.chat.completions.parse(
        model=MODEL,
        messages=messages,
        response_format=RankOrder
    )
    order = response.choices[0].message.parsed
    print(order.order)
    return [chunks[i - 1] for i in order.order]


In [ ]:
RETRIEVAL_K = 10

# def fetch_context_unranked(question):
#     query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
#     results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
#     chunks = []
#     for result in zip(results["documents"][0], results["metadatas"][0]):
#         chunks.append(Result(page_content=result[0], metadata=result[1]))
#     return chunks
def fetch_context_unranked(question):
    query = embedding_model.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
question = "Who was the first human in space?"
chunks = fetch_context_unranked(question)

In [ ]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text retrieved from the company Knowledge Base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with a valid JSON object like {"order":[1,2,3,...]} or a plain comma-separated list of ids, nothing else.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += 'Reply only with a valid JSON object like {"order":[1,2,3,...]} or a plain comma-separated list of ids, nothing else.'
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("{"):
        order = RankOrder.model_validate_json(raw).order
        print(order)
    else:
        raw = raw.strip().strip("[]")
        order = [int(x) for x in raw.replace(" ", "").split(",") if x]
        print(order)

    return [chunks[i - 1] for i in order]


In [ ]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [ ]:
question = "Who was the first human in space?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "RAG" in c.page_content.lower():
        print(index)

In [ ]:
reranked = rerank(question,chunks)

In [ ]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked[0].page_content

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company described in the Knowledge Base.
The Knowledge Base contains the company's identity, capabilities, products, services, and other important facts.
Use the context from the Knowledge Base as your primary source of truth.
If the answer is contained in the context, answer directly from it.
If the answer is not contained in the context, say that you don't know rather than inventing details.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']} (type: {chunk.metadata.get('type', 'unknown')}):\n{chunk.page_content}"
        for chunk in chunks
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]


In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company described in the Knowledge Base.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content about the company.
Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": message}]
    )
    return response.choices[0].message.content


In [ ]:
rewrite_query("Who was the first human in space?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who discovered the Lost Library of Arkania?", [])

In [ ]:
answer_question("hii", [])

In [ ]:
answer_question("what can you do?", [])

In [ ]:
answer_question("What was Ethan's favorite color?", [])

In [ ]:
import gradio as gr

def chat(message, history):
    answer, _ = answer_question(message, history)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="📚 RAG Knowledge Assistant",
    description="Ask questions about the documents in the knowledge base.",
    chatbot=gr.Chatbot(height=500),
    textbox=gr.Textbox(
        placeholder="Ask a question...",
        container=False,
        scale=7
    ),
    # theme=gr.themes.Soft(),
)

demo.launch()